Daniel Lillard

2026.11.06

The 'Hello World' of CLIP. Which was an effective tool for creating new embeddings for the COSC 545 vine project.

In [2]:
import torch
import clip
from PIL import Image

device = "cuda" if torch.cuda.is_available() else "cpu"
model, preprocess = clip.load("ViT-B/32", device=device)

image = preprocess(Image.open("CLIP.png")).unsqueeze(0).to(device)
text = clip.tokenize(["a diagram", "a dog", "a cat"]).to(device)

with torch.no_grad():
    image_features = model.encode_image(image)
    text_features = model.encode_text(text)
    
    logits_per_image, logits_per_text = model(image, text)
    probs = logits_per_image.softmax(dim=-1).cpu().numpy()

print("Label probs:", probs)  # prints: [[0.9927937  0.00421068 0.00299572]]

c:\file_struct\programs\python\Lib\site-packages\torch\jit\_serialization.py:176: FutureWarning: `torch.jit.load` is deprecated. Please switch to `torch.export`.
  warnings.warn(


Label probs: [[0.99279356 0.00421071 0.00299575]]


In [ ]:
# takes a pretty long time to run. Works as expected.

import os
import clip
import torch
from torchvision.datasets import CIFAR100

# Load the model
device = "cuda" if torch.cuda.is_available() else "cpu"
model, preprocess = clip.load('ViT-B/32', device)

# Download the dataset
cifar100 = CIFAR100(root=os.path.expanduser("~/.cache"), download=True, train=False)

# Prepare the inputs
image, class_id = cifar100[3637]
image_input = preprocess(image).unsqueeze(0).to(device)
text_inputs = torch.cat([clip.tokenize(f"a photo of a {c}") for c in cifar100.classes]).to(device)

# Calculate features
with torch.no_grad():
    image_features = model.encode_image(image_input)
    text_features = model.encode_text(text_inputs)

# Pick the top 5 most similar labels for the image
image_features /= image_features.norm(dim=-1, keepdim=True)
text_features /= text_features.norm(dim=-1, keepdim=True)
similarity = (100.0 * image_features @ text_features.T).softmax(dim=-1)
values, indices = similarity[0].topk(5)

# Print the result
print("\nTop predictions:\n")
for value, index in zip(values, indices):
    print(f"{cifar100.classes[index]:>16s}: {100 * value.item():.2f}%")

100%|██████████| 169M/169M [25:55<00:00, 109kB/s]  



Top predictions:

           snake: 65.31%
          turtle: 12.29%
    sweet_pepper: 3.83%
          lizard: 1.88%
       crocodile: 1.75%


In [3]:
# Now, let use do a few thumbnails from the vine dataset.


vine_thumbnails_dir = "C:\\file_struct\\documents\\utk_omscs\\cosc545_digital_archeology\\final_project\\raw_files\\thumbs"

dog_thumbnail = "C:\\file_struct\\documents\\utk_omscs\\cosc545_digital_archeology\\final_project\\raw_files\\thumbs\\0224505EE7.png"



device = "cuda" if torch.cuda.is_available() else "cpu"
model, preprocess = clip.load("ViT-B/32", device=device)

image = preprocess(Image.open(dog_thumbnail)).unsqueeze(0).to(device)
text = clip.tokenize(["a diagram", "a dog", "a cat"]).to(device)

with torch.no_grad():
    image_features = model.encode_image(image)
    text_features = model.encode_text(text)
    
    logits_per_image, logits_per_text = model(image, text)
    probs = logits_per_image.softmax(dim=-1).cpu().numpy()

print("Label probs:", probs)  # predicts a dog with 98% prob.

c:\file_struct\programs\python\Lib\site-packages\torch\jit\_serialization.py:176: FutureWarning: `torch.jit.load` is deprecated. Please switch to `torch.export`.
  warnings.warn(


Label probs: [[0.00235058 0.98253757 0.01511184]]


In [ ]:
# cool! now running on a subset of the data.

# it will take about 3 hours for this to run on all the thumbnails on a CPU
import os
import time
import torch
import clip
from PIL import Image

files = os.listdir(vine_thumbnails_dir)

print(f"Found {len(files)} files in {vine_thumbnails_dir}")

files = files[:100]  # limit to first 10 files for testing

device = "cuda" if torch.cuda.is_available() else "cpu"
model, preprocess = clip.load("ViT-B/32", device=device)

start_time = time.time()

for file in files:
    file_path = os.path.join(vine_thumbnails_dir, file)
    image = preprocess(Image.open(file_path)).unsqueeze(0).to(device)
    text = clip.tokenize(["a person", "a dog", "a cat", "other"]).to(device)

    with torch.no_grad():
        image_features = model.encode_image(image)
        text_features = model.encode_text(text)
        
        logits_per_image, logits_per_text = model(image, text)
        probs = logits_per_image.softmax(dim=-1).cpu().numpy()

    print(label_probs := f"Label probs for {file}: {probs}")


end_time = time.time()
print(f"Processed {len(files)} files in {end_time - start_time:.2f} seconds")

Found 28355 files in C:\file_struct\documents\utk_omscs\cosc545_digital_archeology\final_project\raw_files\thumbs


c:\file_struct\programs\python\Lib\site-packages\torch\jit\_serialization.py:176: FutureWarning: `torch.jit.load` is deprecated. Please switch to `torch.export`.
  warnings.warn(


Label probs for 00DE549D20.png: [[0.5330372  0.07172824 0.02337009 0.3718645 ]]
Label probs for 00DE66AB54.png: [[0.41714644 0.02530028 0.01106849 0.54648477]]
Label probs for 00DEE816AE.png: [[0.36138216 0.05057588 0.04459128 0.54345065]]
Label probs for 00DEF2A5BA.png: [[0.12658706 0.05770661 0.02483664 0.7908697 ]]
Label probs for 00DF2D1D65.png: [[0.0685228  0.02679719 0.01446317 0.8902169 ]]
Label probs for 00E009D808.png: [[0.29066658 0.02736095 0.01600917 0.6659633 ]]
Label probs for 00E03ABF48.png: [[0.24340644 0.05254655 0.0092551  0.69479185]]
Label probs for 00E0713623.png: [[0.22191514 0.03441955 0.00867435 0.734991  ]]
Label probs for 00E0D97D61.png: [[0.17540191 0.30208105 0.4155346  0.10698243]]
Label probs for 00E11A941B.png: [[0.00447554 0.9811624  0.00545738 0.00890462]]
Label probs for 00E11B5C20.png: [[0.7556829  0.01356713 0.01929282 0.21145712]]
Label probs for 00E12214FE.png: [[0.46552414 0.0268234  0.02400118 0.48365137]]
Label probs for 00E13E98CC.png: [[0.2424